<h1><font color='blue'>Análise de Sentimentos e Sistemas de Recomendação</font></h1>
<h2><b>Prática U1-T2-V1.2 - Abordagens de análise de sentimentos</b></h2>


---

## Preparação

### Carregando o dataset

In [6]:
import pandas as pd
df = pd.read_csv('https://drive.google.com/uc?export=download&id=1M7VtNfEh7K3gqa4OiWiayPkg8q6x8fXk', encoding='utf-8', low_memory=False)
df.head()

,frase_pt,frase_en,sentimento
0,O atendimento ao cliente foi excepcional e res...,The customer service was exceptional and solve...,Positivo
1,"Este é, sem dúvida, o melhor café que já prove...","This is, without a doubt, the best coffee I've...",Positivo
2,O filme tem uma fotografia incrível e uma tril...,The movie has incredible cinematography and a ...,Positivo
3,Estou muito satisfeito com a qualidade do prod...,I am very satisfied with the product's quality...,Positivo
4,A equipe foi extremamente simpática e prestati...,The staff was extremely friendly and helpful t...,Positivo


In [7]:
for index, row in df.iterrows():
    print(f"Frase: {row['frase_pt']:<80} - sentimento: {row['sentimento']}")


Frase: O atendimento ao cliente foi excepcional e resolveu meu problema rapidamente.    - sentimento: Positivo
Frase: Este é, sem dúvida, o melhor café que já provei na cidade!                       - sentimento: Positivo
Frase: O filme tem uma fotografia incrível e uma trilha sonora emocionante.             - sentimento: Positivo
Frase: Estou muito satisfeito com a qualidade do produto, superou minhas expectativas.  - sentimento: Positivo
Frase: A equipe foi extremamente simpática e prestativa durante toda a minha visita.    - sentimento: Positivo
Frase: Que experiência maravilhosa! Com certeza voltarei mais vezes.                    - sentimento: Positivo
Frase: O livro chegou em perfeito estado e antes do prazo. Recomendo!                   - sentimento: Positivo
Frase: A vista do quarto do hotel é absolutamente deslumbrante.                         - sentimento: Positivo
Frase: Adorei a nova funcionalidade do aplicativo, ficou muito mais fácil de usar.      - sentimento: Positivo
F

In [8]:
df['sentimento'].value_counts()

,count
sentimento,
Positivo,10
Negativo,10
Neutro,10


### Pré-processamento do texto

In [9]:
import nltk

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [11]:
import re
from nltk.tokenize import word_tokenize
import unicodedata

def preprocess(text):
    text = text.lower()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    return tokens

preprocess('O trânsito      HOJE estava    $$$ %  &  #simplesmente insuportável.')

['o', 'transito', 'hoje', 'estava', 'simplesmente', 'insuportavel']

## Fase 1 — Regras Manuais (Pré-2000)

**Ideia central**

* Listas fixas de palavras positivas e negativas
* Decisão baseada em regras simples

### Definição da lista de palavras positivas e negativas

In [12]:
# lista de palavras
positivas = ["bom", "excelente", "otimo", "gostei", "adorei", "satisfeito"]
negativas = ["ruim", "pessima", "horrivel", "terrivel", "insuportavel"]

# retorna score
def retorna_score(texto: str) -> float:
    score = 0.0
    for palavra in preprocess(texto):
        if palavra in positivas:
            score += 1
        elif palavra in negativas:
            score -= 1
    return score

# função para definir o sentimento
def score_to_sentimento(score: float) -> str:
    if score > 0.0:
        return "Positivo"
    elif score < 0.0:
        return "Negativo"
    else:
        return "Neutro"

### Agregação dos scores

In [13]:
retorna_score('O trânsito hoje estava simplesmente insuportável.')

-1.0

In [14]:
score_to_sentimento(retorna_score('O trânsito hoje estava simplesmente insuportável.'))

'Negativo'

In [15]:
for frase, sentimento in df[['frase_pt','sentimento']].values.tolist():
    score = retorna_score(frase)
    print(f'Frase: {frase:<80} - Sentimento predito: {score_to_sentimento(score)}\t - esperado: {sentimento}\t - Score: {score}')


Frase: O atendimento ao cliente foi excepcional e resolveu meu problema rapidamente.    - Sentimento predito: Neutro	 - esperado: Positivo	 - Score: 0.0
Frase: Este é, sem dúvida, o melhor café que já provei na cidade!                       - Sentimento predito: Neutro	 - esperado: Positivo	 - Score: 0.0
Frase: O filme tem uma fotografia incrível e uma trilha sonora emocionante.             - Sentimento predito: Neutro	 - esperado: Positivo	 - Score: 0.0
Frase: Estou muito satisfeito com a qualidade do produto, superou minhas expectativas.  - Sentimento predito: Positivo	 - esperado: Positivo	 - Score: 1.0
Frase: A equipe foi extremamente simpática e prestativa durante toda a minha visita.    - Sentimento predito: Neutro	 - esperado: Positivo	 - Score: 0.0
Frase: Que experiência maravilhosa! Com certeza voltarei mais vezes.                    - Sentimento predito: Neutro	 - esperado: Positivo	 - Score: 0.0
Frase: O livro chegou em perfeito estado e antes do prazo. Recomendo!           

## Fase 2 — Léxicos de Sentimento (2000–2010)

**Ideia central**

* Uso de dicionários com polaridade e intensidade
* Automatização das regras

### Definição do léxico

In [16]:
lexico = {
    'otimo': 1,
    'bom': 0.8,
    'excelente': 1,
    'incrivel': 1,
    'feliz': 0.9,
    'gostei': 0.7,
    'adorei': 1,
    'satisfeito': 0.8,
    'superou': 0.6,
    'maravilhoso': 1,
    'ruim': -1,
    'terrivel': -1,
    'insuportavel': -1,
    'horrivel': -0.8,
    'chato': -0.5,
    'pessima': -1,
    'odeio': -1,
    'negativa': -0.8,
    'problema': -0.6,
    'nada': 0,
    'neutro': 0,
    'okay': 0.1,
    'bom': 0.5
}


### Agregação dos scores

In [17]:
# retorna score
def retorna_score(texto: str) -> float:
    score = 0.0
    for palavra in preprocess(texto):
        if palavra in lexico:
            score += lexico[palavra]
    return score

In [18]:
retorna_score('Estou muito satisfeito com a qualidade do produto, superou minhas expectativas.')

1.4

In [19]:
for frase, sentimento in df[['frase_pt','sentimento']].values.tolist():
    score = retorna_score(frase)
    print(f'Frase: {frase:<80} - Sentimento predito: {score_to_sentimento(score)}\t - esperado: {sentimento}\t - Score: {score}')


Frase: O atendimento ao cliente foi excepcional e resolveu meu problema rapidamente.    - Sentimento predito: Negativo	 - esperado: Positivo	 - Score: -0.6
Frase: Este é, sem dúvida, o melhor café que já provei na cidade!                       - Sentimento predito: Neutro	 - esperado: Positivo	 - Score: 0.0
Frase: O filme tem uma fotografia incrível e uma trilha sonora emocionante.             - Sentimento predito: Positivo	 - esperado: Positivo	 - Score: 1.0
Frase: Estou muito satisfeito com a qualidade do produto, superou minhas expectativas.  - Sentimento predito: Positivo	 - esperado: Positivo	 - Score: 1.4
Frase: A equipe foi extremamente simpática e prestativa durante toda a minha visita.    - Sentimento predito: Neutro	 - esperado: Positivo	 - Score: 0.0
Frase: Que experiência maravilhosa! Com certeza voltarei mais vezes.                    - Sentimento predito: Neutro	 - esperado: Positivo	 - Score: 0.0
Frase: O livro chegou em perfeito estado e antes do prazo. Recomendo!      

## Fase 3 — Machine Learning Tradicional (2010–2015)

**Ideia central**

* Aprender padrões a partir de dados de exemplo rotulados
* Representar texto numericamente (BoW / TF-IDF)

### Criar a representação numérica do texto com *Bag of Words*

In [20]:
from sklearn.feature_extraction.text import CountVectorizer

vetor = CountVectorizer(lowercase=True)
X = df['frase_pt']
y = df['sentimento']

X_vetor = vetor.fit_transform(X)
pd.DataFrame(X_vetor.toarray(), columns=vetor.get_feature_names_out()).head(10)

,1985,19h,absolutamente,acessar,achei,adaptação,adorei,agendado,ajuda,amassada,...,uma,usar,uso,usuário,vezes,visita,vista,voltarei,voo,às
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,2,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
6,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
8,0,0,0,0,0,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


### Treinar o modelo classificador

In [21]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_vetor, y)


LogisticRegression(max_iter=1000, random_state=42)

### Testar o modelo em novos dados

In [22]:
X_test = vetor.transform(['O atendimento ao cliente foi excepcional e resolveu meu problema rapidamente.'])
pd.DataFrame(X_test.toarray(), columns=vetor.get_feature_names_out()).head(10)

,1985,19h,absolutamente,acessar,achei,adaptação,adorei,agendado,ajuda,amassada,...,uma,usar,uso,usuário,vezes,visita,vista,voltarei,voo,às
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [23]:
model.predict(X_test)

array(['Positivo'], dtype=object)

In [25]:
frases_teste = [
    "O atendimento foi educado, mas demorou mais do que o esperado.",
    "Fiquei impressionado com a rapidez da entrega e o cuidado na embalagem.",
    "O sistema funciona, porém a interface poderia ser mais intuitiva.",
    "Que decepção, esperava muito mais pelo preço que paguei.",
    "O aplicativo cumpriu exatamente o que promete, nada além disso.",
    "A apresentação foi clara e objetiva, facilitando o entendimento do conteúdo.",
    "O quarto estava limpo, mas o barulho externo atrapalhou bastante a estadia.",
    "Excelente custo-benefício, vale cada centavo investido.",
    "O suporte respondeu, mas não resolveu completamente o problema.",
    "Recebi o produto correto, dentro do prazo informado."
]

In [26]:
X_test = vetor.transform(frases_teste)
pd.DataFrame(X_test.toarray(), columns=vetor.get_feature_names_out()).head(10)

,1985,19h,absolutamente,acessar,achei,adaptação,adorei,agendado,ajuda,amassada,...,uma,usar,uso,usuário,vezes,visita,vista,voltarei,voo,às
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [27]:
y_pred = model.predict(X_test)
y_pred

array(['Positivo', 'Neutro', 'Positivo', 'Positivo', 'Negativo',
       'Positivo', 'Negativo', 'Negativo', 'Negativo', 'Positivo'],
      dtype=object)

In [28]:
for frase, sentimento in zip(frases_teste,y_pred):
    print(f'Frase: {frase:<80} - Sentimento predito: {sentimento}')


Frase: O atendimento foi educado, mas demorou mais do que o esperado.                   - Sentimento predito: Positivo
Frase: Fiquei impressionado com a rapidez da entrega e o cuidado na embalagem.          - Sentimento predito: Neutro
Frase: O sistema funciona, porém a interface poderia ser mais intuitiva.                - Sentimento predito: Positivo
Frase: Que decepção, esperava muito mais pelo preço que paguei.                         - Sentimento predito: Positivo
Frase: O aplicativo cumpriu exatamente o que promete, nada além disso.                  - Sentimento predito: Negativo
Frase: A apresentação foi clara e objetiva, facilitando o entendimento do conteúdo.     - Sentimento predito: Positivo
Frase: O quarto estava limpo, mas o barulho externo atrapalhou bastante a estadia.      - Sentimento predito: Negativo
Frase: Excelente custo-benefício, vale cada centavo investido.                          - Sentimento predito: Negativo
Frase: O suporte respondeu, mas não resolveu compl

## Fase 4 — Deep Learning e Transformers (2015-atual)

**Ideia central**

* Modelos pré-treinados
* BERT, GPTs

### Carregar o modelo pré-treinado

In [29]:
!pip install -q pysentimiento

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 32.0 MB/s eta 0:00:00


In [30]:
from pysentimiento import create_analyzer
analyzer = create_analyzer(task="sentiment", lang="pt")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/952 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/562 [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

### Realizar a classificação do sentimento

In [31]:
res = analyzer.predict("Adorei este produto")
res

AnalyzerOutput(output=POS, probas={POS: 0.989, NEU: 0.008, NEG: 0.003})

In [32]:
def retorna_sentimento_py(retorno):
    if retorno == 'NEG':
        return 'Negativo'
    elif retorno == 'NEU':
        return 'Neutro'
    elif retorno == 'POS':
        return 'Positivo'

retorna_sentimento_py(res.output)

'Positivo'

In [33]:
for frase in frases_teste:
    sentimento = retorna_sentimento_py(analyzer.predict(frase).output)
    print(f'Frase: {frase:<80} - Sentimento predito: {sentimento}')


Frase: O atendimento foi educado, mas demorou mais do que o esperado.                   - Sentimento predito: Neutro
Frase: Fiquei impressionado com a rapidez da entrega e o cuidado na embalagem.          - Sentimento predito: Positivo
Frase: O sistema funciona, porém a interface poderia ser mais intuitiva.                - Sentimento predito: Negativo
Frase: Que decepção, esperava muito mais pelo preço que paguei.                         - Sentimento predito: Negativo
Frase: O aplicativo cumpriu exatamente o que promete, nada além disso.                  - Sentimento predito: Positivo
Frase: A apresentação foi clara e objetiva, facilitando o entendimento do conteúdo.     - Sentimento predito: Positivo
Frase: O quarto estava limpo, mas o barulho externo atrapalhou bastante a estadia.      - Sentimento predito: Negativo
Frase: Excelente custo-benefício, vale cada centavo investido.                          - Sentimento predito: Positivo
Frase: O suporte respondeu, mas não resolveu compl